# 10 — TF/TPU Eğitimi: Smoke + GATE + Tier-2 (tf-port branch'i)

**Runtime: TPU seçin** (Çalışma zamanı → Çalışma zamanı türünü değiştir → TPU).
TPU yoksa kod otomatik GPU/CPU'ya düşer (smoke yine çalışır, gate ölçümü TPU ister).

## Ön koşullar (bir kez, ayrı ucuz CPU oturumunda)
1. **Ağırlık export'u** (torch ortamı): repo master'da `python scripts/export_weights_npz.py`
   → `checkpoints/export/*.npz + manifest.json` dosyalarını
   `Drive/AFETSONAR/tf_export/` klasörüne kopyalayın.
2. **TFRecord dönüşümü** (torch'suz): `tf-port` branch'inde
   `python scripts_tf/convert_to_tfrecords.py --csv .../train_v3.csv --split train --copy-paste --out-dir Drive/AFETSONAR/tfrecords`
   (val_v3 ve test_v3 için `--split val` / `--split test`, copy-paste'siz).
3. **Gate referansı** (torch ortamı): `head -n 201 test_v3.csv > test200.csv` →
   `python scripts/evaluate.py --model teacher_v4_best_ema.pth --test-csv test200.csv --output gate_ref.json`
   → `gate_ref.json`'ı `Drive/AFETSONAR/tf_export/`e koyun; aynı `test200.csv` ile
   `convert_to_tfrecords.py --split gate` çalıştırın.

## GATE kriterleri (T4 ancak hepsi yeşilse)
1. TF mIoU_no_bg, 200'lük alt kümede torch referansının **±0.005** içinde
2. 20-epoch maliyet projeksiyonu **≤ ~20 birim**
3. 50 adımda NaN/Inf yok · 4. Checkpoint round-trip başarılı

In [ ]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"  # transformers 4.x TF için şart

!pip install -q "transformers==4.57.6" tf-keras opencv-python-headless

import tensorflow as tf
print("TF:", tf.__version__)

In [ ]:
# Repo (tf-port branch'i) + Drive
if not os.path.exists("/content/calamitas-ai"):
    !git clone -b tf-port https://github.com/Runc-Dev/calamitas-ai.git /content/calamitas-ai
else:
    !cd /content/calamitas-ai && git pull
%cd /content/calamitas-ai
import sys; sys.path.insert(0, "/content/calamitas-ai")

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
DRIVE = Path("/content/drive/MyDrive/AFETSONAR")
EXPORT = DRIVE / "tf_export"     # npz + manifest + gate_ref.json
TFREC  = DRIVE / "tfrecords"     # shard'lar
assert EXPORT.exists(), "tf_export klasörü yok — ön koşul 1'i çalıştırın"
assert TFREC.exists(),  "tfrecords klasörü yok — ön koşul 2'yi çalıştırın"

In [ ]:
# Donanım algılama: TPU -> GPU -> CPU
from afetsonar_tf.training.strategy import detect_strategy
strategy = detect_strategy()
ON_TPU = isinstance(strategy, tf.distribute.TPUStrategy)

if ON_TPU:
    tf.keras.mixed_precision.set_global_policy("mixed_bfloat16")
    print("bfloat16 mixed precision aktif")

In [ ]:
# Shard'ları yerel diske kopyala (Drive I/O eğitimden ayrılır)
import time
t0 = time.time()
!mkdir -p /content/tfrecords /content/export
!cp -n "{TFREC}"/*.tfrecord /content/tfrecords/ 2>/dev/null
!cp -n "{EXPORT}"/*.npz "{EXPORT}"/*.json /content/export/
print(f"Kopyalama: {time.time()-t0:.0f} sn")

import glob
def shards(pattern):
    return sorted(glob.glob(f"/content/tfrecords/{pattern}"))
train_dmg, train_nodmg = shards("train_dmg-*"), shards("train_nodmg-*")
train_cp = shards("train_cp-*")
gate_files = shards("gate_*")
val_files = shards("val_*")
print(f"train: {len(train_dmg)} dmg / {len(train_nodmg)} nodmg / "
      f"{len(train_cp)} cp | gate: {len(gate_files)} | val: {len(val_files)}")

# TF-P5 öz-onarım: gate shard'ları eksikse burada üret (dönüştürücü
# artık --split gate kabul ediyor; test200.csv tf_export'ta duruyor)
if not gate_files:
    print("gate shard'ları yok — 200 görüntü dönüştürülüyor (~2-3 dk)")
    !python scripts_tf/convert_to_tfrecords.py         --csv "{EXPORT}/test200.csv" --split gate --out-dir /content/tfrecords
    !cp /content/tfrecords/gate_* "{TFREC}/"
    gate_files = shards("gate_*")
    print(f"gate shard'ları hazır: {len(gate_files)}")
assert gate_files, "gate shard'ları üretilemedi"


In [ ]:
# Modeli kur + dönüştürülmüş ağırlıkları yükle (strategy scope içinde)
from afetsonar_tf.convert_weights import convert

with strategy.scope():
    model = convert("/content/export", output="")
print("Model hazır — parametre:",
      sum(int(tf.size(v)) for v in model.trainable_variables) / 1e6, "M")

## 1) GATE ölçüm A — 200 görüntülük parite değerlendirmesi

In [ ]:
import json
from afetsonar.evaluation.metrics import SegmentationMetrics
from afetsonar_tf.data import make_eval_dataset

GATE_BATCH = 8  # 200 / 8 = 25 tam batch, örnek düşmez
gate_ds = make_eval_dataset(gate_files, global_batch=GATE_BATCH, size=768)

metrics = SegmentationMetrics(num_classes=6)
dist = strategy.experimental_distribute_dataset(gate_ds)

@tf.function
def predict(image):
    def f(image):
        out = model(image, training=False)
        return tf.argmax(tf.cast(out["damage_logits"][0], tf.float32), -1)
    return strategy.run(f, args=(image,))

n = 0
for image, mask, _ in dist:
    preds = predict(image)
    for p, m in zip(strategy.experimental_local_results(preds),
                    strategy.experimental_local_results(mask)):
        metrics.update(p.numpy(), m.numpy()); n += p.shape[0]

tf_scores = metrics.compute()
ref = json.load(open("/content/export/gate_ref.json"))["metrics"]
diff = abs(tf_scores["miou_no_bg"] - ref["miou_no_bg"])
GATE_PARITY = diff <= 0.005
print(f"{n} görüntü | TF mIoU_no_bg={tf_scores['miou_no_bg']:.4f} | "
      f"torch ref={ref['miou_no_bg']:.4f} | fark={diff:.4f} | "
      f"GATE-A {'YEŞİL' if GATE_PARITY else 'KIRMIZI'}")

## 2) GATE ölçüm B — 50 adımlık eğitim smoke + maliyet projeksiyonu

In [ ]:
import numpy as np, time
from afetsonar.config import DefaultConfig
from afetsonar_tf.data import make_train_dataset
from afetsonar_tf.training.loop import TeacherTrainerTF

GLOBAL_BATCH = 8 * max(strategy.num_replicas_in_sync // 8, 1)
N_TRAIN = 6418
STEPS_PER_EPOCH = N_TRAIN // GLOBAL_BATCH
EPOCHS_PLANNED = 20

cfg = DefaultConfig()
train_ds = make_train_dataset(train_dmg, train_nodmg,
                              global_batch=GLOBAL_BATCH,
                              cp_files=train_cp or None, size=768)

trainer = TeacherTrainerTF(
    model, strategy,
    total_steps=EPOCHS_PLANNED * STEPS_PER_EPOCH,
    peak_lr=1e-5, class_weights=cfg.class_weights,
)

dist_train = strategy.experimental_distribute_dataset(train_ds)
it = iter(dist_train)
losses, times = [], []
for step in range(50):
    image, mask, dis = next(it)
    t0 = time.time()
    loss = trainer._train_step(image, mask, dis)
    times.append(time.time() - t0)
    losses.append(float(loss))
    if (step + 1) % 10 == 0:
        print(f"adım {step+1}: loss={losses[-1]:.4f} süre={times[-1]:.2f}s")

GATE_NAN = all(np.isfinite(losses))
steady = times[5:]  # ilk adımlar XLA derlemesi — sayma
step_s = float(np.median(steady))
hours = EPOCHS_PLANNED * STEPS_PER_EPOCH * step_s / 3600
units = hours * 1.76  # Colab TPU ~1.76 birim/saat
GATE_COST = units <= 20
print(f"\nMedyan adım: {step_s:.2f}s | 20 epoch ≈ {hours:.1f} saat "
      f"≈ {units:.1f} birim | NaN yok: {GATE_NAN} | "
      f"GATE-B {'YEŞİL' if GATE_COST else 'KIRMIZI'}")

In [ ]:
# 3) Checkpoint round-trip (gerçek model)
ck = "/content/ckpt_test/teacher_tf.ckpt"
sample = next(iter(gate_ds))[0][:2]
before = model(sample, training=False)["damage_logits"][0].numpy()
model.save_weights(ck)
model.load_weights(ck)
after = model(sample, training=False)["damage_logits"][0].numpy()
GATE_CKPT = bool(np.allclose(before, after, atol=1e-5))
print("Checkpoint round-trip:", "YEŞİL" if GATE_CKPT else "KIRMIZI")

In [ ]:
# GATE KARARI
GATE_GREEN = GATE_PARITY and GATE_COST and GATE_NAN and GATE_CKPT and ON_TPU
print("="*50)
print(f"Parite ±0.005 : {GATE_PARITY}")
print(f"Maliyet ≤20 b. : {GATE_COST}")
print(f"NaN/Inf yok    : {GATE_NAN}")
print(f"Checkpoint     : {GATE_CKPT}")
print(f"TPU üzerinde   : {ON_TPU}")
print(f"GATE: {'🟢 YEŞİL — Tier-2 başlatılabilir' if GATE_GREEN else '🔴 KIRMIZI — T4 başlatmayın, sonucu raporlayın'}")

## 4) Tier-2 fine-tune (YALNIZCA gate yeşilse)
20 epoch @768, lr 1e-5, Copy-Paste shard'ları, EMA doğrulama.
En iyi EMA checkpoint'i Drive'a yazılır.

In [ ]:
RUN_TIER2 = False  # GATE yeşilse ve karar verdiyseniz True yapın

if RUN_TIER2:
    assert GATE_GREEN, "GATE kırmızı — Tier-2 başlatılamaz (protokol)"
    val_ds = make_eval_dataset(val_files, global_batch=GLOBAL_BATCH, size=768)
    history = trainer.fit(
        train_ds, val_ds,
        epochs=EPOCHS_PLANNED, steps_per_epoch=STEPS_PER_EPOCH,
        ckpt_dir="/content/ckpts",
    )
    !mkdir -p "{DRIVE}/checkpoints/tf"
    !cp /content/ckpts/teacher_tf_best_ema* "{DRIVE}/checkpoints/tf/"
    !cp /content/ckpts/history.json "{DRIVE}/checkpoints/tf/"
    print("En iyi TF checkpoint Drive'a yazıldı")